<a href="https://colab.research.google.com/github/samar99-tech/arabic-AI-text-Detection/blob/main/notebooks/feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
url = "https://raw.githubusercontent.com/samar99-tech/arabic-AI-text-Detection/main/data/processed/processed_data.csv"
df = pd.read_csv(url)
print(df.shape)
print(df["label"].value_counts())

(12866, 7)
label
1    10292
0     2574
Name: count, dtype: int64


In [2]:
#First Feature
#F13: Hapax Legomena Ratio
#unique words ratio: number of words appeared once / total number of words

from collections import Counter

def hapax_ratio(t):
    words = str(t).split()
    if len(words) == 0:
        return 0
    counts = Counter(words)
    hapax = 0
    for w in counts:
        if counts[w] == 1:
            hapax = hapax + 1
    return hapax / len(words)

df["f13_hapax"] = df["text_normalized"].apply(hapax_ratio) #normalized used so punctuation attached to words is not counted as different words
print(df.groupby("label")["f13_hapax"].mean().round(3))


label
0    0.673
1    0.594
Name: f13_hapax, dtype: float64


F13-Hapax ratio: human (0.67) vs AI (0.59). Human texts have a higher ratio than AI, which means they used more words that appear only once (rare words), while AI tends to repeat words. TRR counts the repeated words once unlike Hapax ratio which does not count the at all.

In [3]:
#Second Feature:
#F35: Total number of paragraphs
#how many paragarphs in the texts separated by a line


def count_paragraphs(t):
    lines = str(t).split("\n")
    paragraphs = [p for p in lines if p.strip() != ""]
    return len(paragraphs)

df["f35_paragraphs"] = df["text"].apply(count_paragraphs)
print(df.groupby("label")["f35_paragraphs"].mean().round(3))
print(df.groupby("label")["f35_paragraphs"].value_counts().head(10))

label
0    1.000
1    1.578
Name: f35_paragraphs, dtype: float64
label  f35_paragraphs
0      1                 2574
1      1                 7866
       2                  879
       4                  597
       3                  568
       5                  199
       6                   65
       7                   43
       8                   22
       9                   18
Name: count, dtype: int64


all the 2574 human texts have exactly one paragraph, while 7866 AI texts have one paragraph and the renaining 2426 text have more than one. This can be used as indicator for classifing AI texts. However, if a text has one paragraph it is hard to tell whether it is human or AI. Having all human text in one paragraph may come from the way data was collected; for example, abstracts taken from Journal, may have been saved as one paragraph.

In [4]:
%%capture
!pip install -q transformers


In [5]:
#Third Feature:
#f57: number of words found in the 50 positions within word embedding
#(using the word embedding of a language model, as the instructions require)
#the model AraGPT2 reads the text word by word and ranks all words it expects next
#will count how many real words were inside the model top 50 positions
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
tqdm.pandas()

device = "cuda" if torch.cuda.is_available() else "cpu"
name = "aubmindlab/aragpt2-base"#Arabic language model
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name).to(device).eval()

def top50(t):
    ids = tok(str(t), return_tensors="pt", truncation=True, max_length=512).input_ids.to(device)  # text to tokens
    if ids.shape[1] < 2:
        return 0, 0
    with torch.no_grad():
        scores = model(ids).logits[0, :-1] #model scores for every possible next word
    real = ids[0, 1:] #the real next words in the text
    real_scores = scores.gather(1, real.unsqueeze(1))
    ranks = (scores > real_scores).sum(1) #position of each real word (0 = first)
    count = int((ranks < 50).sum()) #real words inside the top 50 positions
    return count, count / len(real)#count and ratio

res = df["text"].progress_apply(top50)
df["f57_count"] = [r[0] for r in res]
df["f57_ratio"] = [r[1] for r in res]
print(df.groupby("label")[["f57_count", "f57_ratio"]].mean().round(3))

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.52M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  553MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: aubmindlab/aragpt2-base
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/12866 [00:00<?, ?it/s]

       f57_count  f57_ratio
label                      
0        107.451      0.645
1         97.247      0.688


The word embedding of a language model (AraGPT2) was used and it ranks the words it expects at each position. Then I counted how many real words in the text were inside its top 50 positions. AI texts have a higher ratio(0.688) than human texts(0.645) since AI models choose the most expected words, while humans use less predictable words. The human count(107.451) is higher than AI(97.247) mainly because human texts are longer.

In [6]:
#Fourth Feature:
#F79: Burstiness
#does sentences length chage?
#
#between [-1 , 1] where -1 means
import numpy as np

def burstiness(t):
    t = str(t).replace("!", ".").replace("؟", ".").replace("?", ".")
    sentences = [s for s in t.split(".") if s.strip() != ""]
    lengths = [len(s.split()) for s in sentences]
    if len(lengths) < 2:
        return 0
    return np.std(lengths) / np.mean(lengths)


df["f79_burstiness"] = df["text"].apply(burstiness)

print(df.groupby("label")["f79_burstiness"].mean().round(3))

df["n_sentences"] = df["text"].apply(lambda t: len([s for s in str(t).replace("!", ".").replace("؟", ".").replace("?", ".").split(".") if s.strip() != ""]))
print((df[df["n_sentences"] < 2].groupby("label").size()))# if less than 2 (one sentence)

print(df[["label", "n_sentences", "f79_burstiness"]].head(10))

label
0    0.321
1    0.262
Name: f79_burstiness, dtype: float64
label
0    637
1     13
dtype: int64
   label  n_sentences  f79_burstiness
0      0            3        0.950438
1      1            7        0.310197
2      1            6        0.122975
3      1            7        0.167229
4      1            6        0.252092
5      0            1        0.000000
6      1            6        0.320528
7      1            5        0.354645
8      1            3        0.339417
9      1            5        0.071894



Burstiness meaures (how much sentences length change within a text?) computed as std /mean of sentence lengths. 0 means all sentences have the same length. higher values means more variation.
Human texts score higher (0.321) than AI texts(0.262), so humans mix long and short sentences while AI write in similar length. Human has 637 texts with only one sentence(score =0) and this lower the average whereas AI has 13 texts with one sentence. it shows that real difference is larger.

In [7]:
#Fifth Feature
#F101: Diacritic Presence Ratio
#number of diacritics / number of Arabic letters
import re

def diacritic_ratio(t):
    t = str(t)
    diacritized = len(re.findall(r"[\u0621-\u064A][\u064B-\u0652]", t))
    if len(t) == 0:
        return 0
    return diacritized / len(t)

df["f101_diacritics"] = df["text"].apply(diacritic_ratio)
print(df.groupby("label")["f101_diacritics"].mean().round(4))

#how many texts have at least one diacritic
print((df["f101_diacritics"] > 0).groupby(df["label"]).mean().round(3))

label
0    0.0051
1    0.0042
Name: f101_diacritics, dtype: float64
label
0    0.401
1    0.743
Name: f101_diacritics, dtype: float64


both classes use very little tashkeel (human 0.0051, AI 0.0042). However, 74% of AI texts contain at least one diacritic, compared with 40% of human texts. AI adds a few diacritics in many texts, while humans add them in fewer texts but use more when they do.

In [8]:
#save all features to a file
features = df[["text", "label", "source", "f13_hapax", "f35_paragraphs", "f57_count", "f57_ratio", "f79_burstiness", "f101_diacritics"]]
features.to_csv("features.csv", index=False, encoding="utf-8-sig")
print(features.shape)

(12866, 9)


In [9]:
#split data into train (70%), validation (15%) and test (15%)

from sklearn.model_selection import train_test_split

train, temp = train_test_split(features, test_size=0.30, stratify=features["label"], random_state=42)
val, test = train_test_split(temp, test_size=0.50, stratify=temp["label"], random_state=42)

print("train:", train.shape, "val:", val.shape, "test:", test.shape)
print(train["label"].value_counts(normalize=True).round(3))

train.to_csv("train.csv", index=False, encoding="utf-8-sig")
val.to_csv("val.csv", index=False, encoding="utf-8-sig")
test.to_csv("test.csv", index=False, encoding="utf-8-sig")

train: (9006, 9) val: (1930, 9) test: (1930, 9)
label
1    0.8
0    0.2
Name: proportion, dtype: float64


The data was split into training (70%, 9006), valdiation (15%, 1930) and test (15%, 1930).The split keeps the same class proprtions (80% AI, 20% human) in every set.

In [10]:
from google.colab import files
files.download("features.csv")
files.download("train.csv")
files.download("val.csv")
files.download("test.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>